# Phase 1 — Data + Environment

This notebook:
1. Installs dependencies
2. Mounts Google Drive (dataset persists here across sessions)
3. Downloads THINGS-EEG via `osfclient`
4. Clones/updates the EEG-FineTune repo so `src/dataset.py` is importable
5. Runs sanity check — stats, NaN/Inf assert, 5-epoch plot
6. Builds CLIP embedding cache (once)

**Run cells top-to-bottom. Re-running is safe — downloads and caches are skipped if already present.**

## Step 1 — Install dependencies

In [1]:
import importlib, subprocess, sys, os

PKGS = ["mne>=1.7.0", "open-clip-torch>=2.24.0", "osfclient>=0.0.5", "tqdm>=4.66.0"]

# Check if osfclient is already installed (i.e. we already restarted once)
already_installed = importlib.util.find_spec("osfclient") is not None

if not already_installed:
    print("Installing packages — runtime will restart automatically afterwards...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + PKGS)
    print("Done. Restarting runtime so CLI tools are on PATH...")
    os.kill(os.getpid(), 9)  # triggers Colab auto-restart
else:
    print("Packages already installed — no restart needed.")

ERROR: Could not find a version that satisfies the requirement osfclient>=0.0.12 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5)
ERROR: No matching distribution found for osfclient>=0.0.12


## Step 2 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/EEG-FineTune')
DATA_ROOT    = PROJECT_ROOT / 'data' / 'things-eeg'
REPO_DIR     = Path('/content/EEG-FineTune')

DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DATA_ROOT    : {DATA_ROOT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT : /content/drive/MyDrive/EEG-FineTune
DATA_ROOT    : /content/drive/MyDrive/EEG-FineTune/data/things-eeg


## Step 3 — Clone / update the EEG-FineTune repo

In [3]:
import subprocess, sys

# Replace with your GitHub repo URL after first push
REPO_URL = 'https://github.com/stephanieolaiya/EEG-FineTune.git'

if REPO_DIR.exists():
    print('Repo already cloned — pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    print(f'Cloning {REPO_URL}...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready at', REPO_DIR)

Repo already cloned — pulling latest...
Repo ready at /content/EEG-FineTune


## Step 4 — Download THINGS-EEG dataset

Hosted on OSF: https://osf.io/3jk45/ (~15-20 GB total).  
Files already on Drive are skipped automatically.

In [4]:
import requests, shutil, zipfile
from pathlib import Path

# Direct OSF download URLs for THINGS-EEG2 preprocessed data
# Source: OSF child node 'anp5v' (Preprocessed EEG data) under project 3jk45
SUBJECT_ZIPS = {
    '01': 'https://osf.io/download/nb8wr/',
    '02': 'https://osf.io/download/u43p5/',
    '03': 'https://osf.io/download/fy4n6/',
    '04': 'https://osf.io/download/py8jz/',
    '05': 'https://osf.io/download/2cn7v/',
    '06': 'https://osf.io/download/bd83f/',
    '07': 'https://osf.io/download/mw8hr/',
    '08': 'https://osf.io/download/n9bzm/',
    '09': 'https://osf.io/download/fv6jz/',
    '10': 'https://osf.io/download/8fyu9/',
}
SUBJECTS = list(SUBJECT_ZIPS.keys())

# VM local staging — large files must land here first before copying to Drive
VM_STAGING = Path('/content/things-eeg2-staging')
VM_STAGING.mkdir(parents=True, exist_ok=True)

def stream_download(url, dest_path, desc=''):
    """Stream a file from url to dest_path, show MB progress."""
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'  Downloading {desc} ...', end=' ', flush=True)
    with requests.get(url, stream=True, timeout=120, allow_redirects=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        written = 0
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=4 << 20):  # 4 MB chunks
                f.write(chunk)
                written += len(chunk)
        mb = dest_path.stat().st_size / 1e6
    print(f'{mb:.0f} MB ✓')
    return dest_path

def download_and_extract_subject(sub):
    url       = SUBJECT_ZIPS[sub]
    zip_stage = VM_STAGING / f'sub-{sub}.zip'
    out_dir   = DATA_ROOT / 'preprocessed_data' / f'sub-{sub}'

    # Check if already extracted to Drive
    train_file = out_dir / 'preprocessed_eeg_training.npy'
    test_file  = out_dir / 'preprocessed_eeg_test.npy'
    if train_file.exists() and test_file.exists():
        print(f'  [skip] sub-{sub} already on Drive')
        return

    # Download ZIP to VM if needed
    if not zip_stage.exists() or zip_stage.stat().st_size < 1e8:
        stream_download(url, zip_stage, desc=f'sub-{sub}.zip')

    # Extract ZIP on VM
    print(f'  Extracting sub-{sub}.zip ...', end=' ', flush=True)
    vm_extract = VM_STAGING / f'sub-{sub}'
    with zipfile.ZipFile(zip_stage, 'r') as zf:
        zf.extractall(vm_extract)
    print('done')

    # Find the .npy files inside the extracted tree and copy to Drive
    out_dir.mkdir(parents=True, exist_ok=True)
    found = list(vm_extract.rglob('preprocessed_eeg_training.npy'))
    if not found:
        print(f'  [ERROR] preprocessed_eeg_training.npy not found inside ZIP. Files:')
        print('  ' + '\n  '.join(str(p) for p in vm_extract.rglob('*')))
        return
    for src in vm_extract.rglob('preprocessed_eeg_*.npy'):
        dst = out_dir / src.name
        shutil.copy2(src, dst)
        print(f'  -> Drive: {dst.name} ({dst.stat().st_size / 1e6:.0f} MB) ✓')

print('Downloading THINGS-EEG2 preprocessed data...')
print(f'Subjects: {SUBJECTS}\n')
for sub in SUBJECTS:
    print(f'sub-{sub}:')
    download_and_extract_subject(sub)

print('\nNote: stimulus image_set/ (~6 GB) needed for CLIP cache (Step 6).')
print('\nDownload step complete.')

  Fetching osfstorage/image_metadata.npy...


FileNotFoundError: [Errno 2] No such file or directory: 'osf'

## Step 5 — Sanity check

In [ ]:
import numpy as np
import pprint

sub = '01'
eeg_path = DATA_ROOT / 'preprocessed_data' / f'sub-{sub}' / 'preprocessed_eeg_training.npy'

if not eeg_path.exists():
    raise FileNotFoundError(f'EEG file not found: {eeg_path}\nMake sure Step 4 completed successfully.')

# THINGS-EEG2 files are dicts: {'preprocessed_eeg_data', 'ch_names', 'times'}
d     = np.load(eeg_path, allow_pickle=True).item()
eeg   = d['preprocessed_eeg_data']   # (n_concepts, n_reps, n_channels, n_times)
times = d['times']
sfreq = round(1.0 / float(np.diff(times).mean()))

n_concepts, n_reps, n_channels, n_times = eeg.shape

print('=' * 55)
print('DATASET STATISTICS  —  sub-01 / training')
print('=' * 55)
pprint.pprint({
    'n_concepts'        : n_concepts,
    'n_repetitions'     : n_reps,
    'n_channels'        : n_channels,
    'n_times (samples)' : n_times,
    'sampling_rate_hz'  : sfreq,
    'time_range_s'      : f'{times[0]:.2f} to {times[-1]:.2f}',
    'dtype'             : str(eeg.dtype),
    'shape'             : eeg.shape,
    'value_range'       : f'[{eeg.min():.4f}, {eeg.max():.4f}]',
    'ch_names[:5]'      : d['ch_names'][:5],
}, sort_dicts=False)

print()
has_nan = bool(np.any(np.isnan(eeg)))
has_inf = bool(np.any(np.isinf(eeg)))
if has_nan or has_inf:
    raise ValueError(f'Data quality check FAILED — NaN: {has_nan}, Inf: {has_inf}')
print('[PASS] No NaN or Inf values found.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.dataset import zscore_normalize, crop_epoch

rng = np.random.default_rng(42)
concept_indices = rng.choice(n_concepts, size=5, replace=False)

# Crop window: 0 to 0.5 s post-stimulus
t_mask    = (times >= 0.0) & (times < 0.5)
time_axis = times[t_mask] * 1000  # convert to ms

fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
fig.suptitle(
    '5 Random EEG Epochs  (sub-01, rep-averaged, z-score, 0–500 ms)',
    fontsize=12
)

for ax, c_idx in zip(axes, concept_indices):
    # Average across repetitions, then preprocess
    epoch = eeg[c_idx].mean(axis=0).copy()  # (n_channels, n_times)
    epoch = zscore_normalize(epoch)
    epoch = crop_epoch(epoch, times, t_start=0.0, duration=0.5)

    for ch in range(epoch.shape[0]):
        ax.plot(time_axis, epoch[ch], linewidth=0.4, alpha=0.5, color='steelblue')

    ax.set_ylabel(f'Concept {c_idx}\n(z-score)', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_ylim(-5, 5)

axes[-1].set_xlabel('Time (ms post-stimulus)')
plt.tight_layout()

out_path = PROJECT_ROOT / 'sanity_check_epochs.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Plot saved to {out_path}')
plt.show()

## Step 6 — Build CLIP embedding cache

Runs once (~10-20 min on T4). Saves `embeddings_cache.npy` to Drive.

In [ ]:
import torch
import numpy as np
from src.dataset import build_clip_embedding_cache

image_set_dir = DATA_ROOT / 'image_set'
cache_path    = DATA_ROOT / 'embeddings_cache.npy'

if not image_set_dir.exists():
    print(f'[SKIP] image_set/ not found at {image_set_dir}')
    print('Download stimulus images from OSF first, then re-run this cell.')
elif cache_path.exists():
    emb = np.load(cache_path)
    print(f'[SKIP] Cache already exists  shape={emb.shape}')
else:
    meta = np.load(DATA_ROOT / 'image_metadata.npy', allow_pickle=True).item()
    image_files = list(meta['image_path'])
    full_paths  = [str(image_set_dir / f) for f in image_files]
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Using device: {device}')
    emb = build_clip_embedding_cache(full_paths, cache_path, device=device)
    print(f'Done. shape={emb.shape}')

## Phase 1 complete

If all cells above passed:
- Stats printed without errors
- `[PASS] No NaN or Inf values found` appeared
- Epoch plot saved to `MyDrive/EEG-FineTune/sanity_check_epochs.png`

Proceed to **Phase 2 — Signal Encoder**.